In [71]:
import numpy as np
import math
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [72]:
X, y = load_breast_cancer(return_X_y=True)

In [73]:
X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size = 0.3,random_state = 42)
X_val,X_test,y_val,y_test = train_test_split(X_temp,y_temp,test_size = 0.5,random_state = 42)

In [76]:
def zscore_normalize_features(X, mu=None, sigma=None):
    if mu is None:
        mu = np.mean(X, axis=0)
        sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

In [77]:
X_train_norm, mu, sigma = zscore_normalize_features(X_train)
X_val_norm, _, _  = zscore_normalize_features(X_val, mu, sigma)
X_test_norm, _, _ = zscore_normalize_features(X_test, mu, sigma) 

In [78]:
def sigmoid(z):
    return np.where(z>=0 , 1/(1 + np.exp(-z)), np.exp(z)/(1 + np.exp(z)))

In [79]:
def compute_cost(X, y, w, b):
    m, n = X.shape
    cost = 0
    eps = 1e-15
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb = sigmoid(z_i)
        f_wb = np.clip(f_wb, eps, 1 - eps)
        cost += -1*(y[i]*np.log(f_wb) + (1-y[i])*np.log(1-f_wb))
    return cost/m

In [80]:
def compute_gradient(X, y, w, b):
    m, n = X.shape
    dj_dw = np.zeros(n)
    dj_db = 0
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb = sigmoid(z_i)
        err = f_wb - y[i]
        for j in range(n):
            dj_dw[j] += err*X[i, j]
        dj_db += err
    return dj_db/m, dj_dw/m

In [81]:
def gradient_descent(X, y, w, b, cost_function, gradient_function, alpha, Iter):
    J_history = []
    w_history = []
    for i in range(Iter):
        dj_db, dj_dw = gradient_function(X, y, w, b)
        w = w - alpha*dj_dw
        b = b - alpha*dj_db
        if i < 100000:
            J_history.append(cost_function(X, y, w, b))
        if i % math.ceil(Iter/10) == 0 or i == (Iter-1):
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}")
    return w, b, J_history, w_history

In [83]:
w_init = np.zeros(X_train_norm.shape[1])
b_init = 0.0
alpha = 0.1
iters = 10000

w, b, J_hist, w_hist = gradient_descent(
    X_train_norm, y_train, w_init, b_init,
    compute_cost, compute_gradient, alpha, iters
)

Iteration    0: Cost     0.53
Iteration 1000: Cost     0.06
Iteration 2000: Cost     0.06
Iteration 3000: Cost     0.05
Iteration 4000: Cost     0.05
Iteration 5000: Cost     0.05
Iteration 6000: Cost     0.05
Iteration 7000: Cost     0.05
Iteration 8000: Cost     0.05
Iteration 9000: Cost     0.04
Iteration 9999: Cost     0.04


In [90]:
def predict(X, w, b, threshold=0.5):
    probs = sigmoid(np.dot(X, w) + b)
    return (probs >= threshold).astype(int)

In [91]:
train_acc = np.mean(predict(X_train_norm, w, b) == y_train)
val_acc   = np.mean(predict(X_val_norm, w, b) == y_val)

print(f"\nTrain accuracy: {train_acc:.4f}")
print(f"Val accuracy:   {val_acc:.4f}")


Train accuracy: 0.9899
Val accuracy:   0.9882
